In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import shutil
from pathlib import Path
import pandas as pd
from tqdm import tqdm

sys.path.append(r'E:\repository\dataset_tools\isds_tool\PS_data')
from google_drive_tools import download_all_subfolders_parallel
from zip_tools import uzip_dirs, zip_folder_to_path, uzip_file
from yolo_tools import att_check, find_defect, copy_all

In [3]:

root_dir = r'\\158.132.186.40\isds\huilin\isds\google_drive_data\task1006'
merge_dir = os.path.join(root_dir, 'merge_dir')
root_folder_id = '1vVYAVjTqoy_WYP8Zro5HzjHsPaipRK2F'
client_secret = r"E:\data\202502_signboard\data_annotation\docs\client_secret.json"
token_path = r'E:\repository\dataset_tools\isds_tool\PS_data\token.json'

gap_num = 3

In [4]:
# os.remove(token_path)

In [5]:
download_all_subfolders_parallel(token_path, client_secret, root_folder_id, root_dir)

将并发下载 2 个子文件夹...


📁 Starting folder: 03-52-15_PM

📁 Starting folder: 03-21-22_PM
⚠️ 已存在，跳过: \\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\03-52-15_PM\gps_data.html
⚠️ 已存在，跳过: \\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\03-52-15_PM\slam_geo_referenced_snapped.txt
⚠️ 已存在，跳过: \\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\03-52-15_PM\slam_geo_referenced_snapped_manual.html
⚠️ 已存在，跳过: \\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\03-52-15_PM\15-52-15.pcd
⚠️ 已存在，跳过: \\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\03-52-15_PM\gps_data.txt
⚠️ 已存在，跳过: \\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\03-52-15_PM\slam_geo_referenced.html
⚠️ 已存在，跳过: \\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\03-52-15_PM\step2_converted_gps_hk1980.png
⚠️ 已存在，跳过: \\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\03-52-15_PM\geo_ref_matrix.txt
⚠️ 已存在，跳过: \\158.132.186.40\isds\huilin\isds\google_drive_d

In [6]:
# uzip_dirs(root_dir, zip_relative_path='cameras.zip')

In [7]:
from img_preprocess import select_img
from deduplication_demo import filter_deduplication

def process_dirs(root_dir):
    sub_dirs = os.listdir(root_dir)
    for idx, sub_name in enumerate(sub_dirs):
        sub_dir = os.path.join(root_dir, sub_name)
        if not os.path.isdir(sub_dir) or sub_name.startswith('r'):
            continue
        cam_name_list = ['camera1', 'camera2', 'camera3', 'camera4', 'camera5', 'camera6']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, 'cameras', cam_name)
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                print(f'{image_dir_src} selecting...')
                image_dir_select = image_dir_src+'_select'
                shutil.rmtree(image_dir_select) if os.path.exists(image_dir_select) else None
                select_img(image_dir_src, image_dir_select, gap=gap_num)
                print(f'{image_dir_select} filtering...')
                image_dir_filter = image_dir_src+'_filter'
                shutil.rmtree(image_dir_filter) if os.path.exists(image_dir_filter) else None
                filter_deduplication(image_dir_select, image_dir_filter)
                print(f'{image_dir_filter} done\n')

In [ ]:
process_dirs(root_dir)

\\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\03-21-22_PM\cameras\camera1 selecting...


100%|██████████| 2488/2488 [03:00<00:00, 13.80it/s]


\\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\03-21-22_PM\cameras\camera1_select filtering...


 48%|████▊     | 1204/2488 [02:56<03:02,  7.05it/s]

In [ ]:
def img_merge(input_dir, output_dir):
    sub_dirs = os.listdir(input_dir)
    if 'merge_dir' in sub_dirs:
        sub_dirs.remove('merge_dir')
    os.makedirs(output_dir, exist_ok=True)
    for sub_name in sub_dirs:
        sub_dir = os.path.join(input_dir, sub_name)
        if not os.path.isdir(sub_dir) or sub_name.startswith('r'):
            continue
        cam_name_list = ['camera1', 'camera2', 'camera3', 'camera4', 'camera5', 'camera6']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, 'cameras', cam_name+'_filter')
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                img_list = os.listdir(image_dir_src)
                for img_name in tqdm(img_list):
                    img_path_src = os.path.join(image_dir_src, img_name)
                    img_path_dst = os.path.join(output_dir, cam_name+'_'+img_name)
                    shutil.copyfile(img_path_src, img_path_dst)

In [ ]:
img_merge(root_dir, merge_dir)

In [ ]:
print(len(os.listdir(merge_dir)))

In [ ]:
zip_folder_to_path(
    source_folder=merge_dir,
    destination_zip=os.path.join(root_dir, os.path.basename(root_dir)+'.zip')
)

In [ ]:
zip_folder_to_path(
    source_folder=os.path.join(merge_dir+'_infer', 'labels'),
    destination_zip=os.path.join(root_dir, os.path.basename(root_dir)+'_labels.zip')
)

In [ ]:
file_list = os.listdir(root_dir)
new_label_zip = None
for file_name in file_list:
    if file_name.endswith('.zip') and len(file_name.split('_')) == 3:
        new_label_zip = os.path.join(root_dir, file_name)
        break

In [ ]:
new_label_dir = os.path.join(root_dir, os.path.basename(new_label_zip).split('.')[0])
uzip_file(new_label_zip, new_label_dir)

\\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\task1006_labels_1015.zip unzip...
\\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\task1006_labels_1015.zip done



In [ ]:
full_new_label_dir = os.path.join(new_label_dir, os.path.basename(new_label_dir))
check_label_dir = os.path.join(new_label_dir, 'check_labels')
select_dir = os.path.join(root_dir, 'select')
select_label_dir = os.path.join(select_dir, 'labels')
select_image_dir = os.path.join(select_dir, 'images')
defect_list = ['deformation', 'broken', 'abandonment', 'corrosion']
att_check(full_new_label_dir, check_label_dir, reorder=False, rm_id=True)
find_defect(check_label_dir, select_label_dir, merge_dir, select_image_dir, defect_list)

att_check: 100%|██████████| 1005/1005 [00:14<00:00, 71.09it/s]


change "0" 0 lines
remove 41 id


find_defect: 100%|██████████| 1005/1005 [00:07<00:00, 127.24it/s]

find 46 files with defect


In [ ]:
fused_dir = r'\\158.132.186.40\isds\huilin\isds\fused_data\task1015'
fused_label_dir = os.path.join(fused_dir, 'labels')
fused_image_dir = os.path.join(fused_dir, 'images')
copy_all(select_label_dir, fused_label_dir, select_image_dir, fused_image_dir)

copy_all: 100%|██████████| 46/46 [00:02<00:00, 20.34it/s]


In [ ]:
print(f'find {len(os.listdir(select_image_dir))} in {len(os.listdir(merge_dir))}')
print(f'{select_image_dir}\n{merge_dir}')

find 46 in 1137
\\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\select\images
\\158.132.186.40\isds\huilin\isds\google_drive_data\task1006\merge_dir
